# 04 – M3: BiLSTM + max-pooling (W3-01)

Cần chạy `00_colab_setup.ipynb` và phần W2-08 của `03_m2.ipynb` ít nhất một lần trước (để có dữ liệu và `emb_fasttext.npz` trên Drive).
Nên chọn GPU: **Runtime → Change runtime type → T4 GPU** (CPU cũng chạy được: S2 vài phút, S1 khoảng 1 giờ).
Checkpoint lưu vào Drive chung, thư mục `NLI/checkpoints/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

## Kiểm tra nhanh: M3 phân biệt được thứ tự từ (M2 thì không)
Vector câu của "Việt Nam thắng Thái Lan" và "Thái Lan thắng Việt Nam": ở M2 (trung bình) giống hệt nhau, ở M3 khác nhau.

In [ ]:
import numpy as np, torch
from src import m3
from src.embeddings import load_embedding
w2i, M = load_embedding()
enc = m3.M3(M, m3.DEFAULT_CONFIG).encoder.eval()
a, b = 'Việt Nam thắng Thái Lan', 'Thái Lan thắng Việt Nam'
x, l = m3.pad_batch(m3.encode([a, b], w2i, 64))
with torch.no_grad():
    u = enc(x, l)
avg = lambda s: M[[w2i.get(t, 0) for t in s.split()]].mean(0)
print('M2 – hai vector giống nhau:', np.allclose(avg(a), avg(b)))
print('M3 – hai vector giống nhau:', torch.allclose(u[0], u[1]))

## Chọn siêu tham số trên dev (không dùng test)
S2 đã chọn xong (kết quả trong `results/analysis/m3_tuning_S2.json`), **bỏ qua ô này** nếu không muốn chạy lại.

In [ ]:
!python scripts/tune_m3.py --scenario S2

S1: chỉ thử đóng băng hay không đóng băng embedding (S1 lớn, chạy lâu). Khoảng 10–20 phút trên GPU.

In [ ]:
!python scripts/tune_m3.py --scenario S1 --grid freeze_embedding

## W3-01: chạy M3 cho S2 và S1 (seed 42), cấu hình lấy từ `m3.config_for`

In [ ]:
!python scripts/run_m3.py --scenario S2 S1 --seeds 42

## Kết quả

In [ ]:
from src.evaluate import summarize
from IPython.display import Image, display
display(summarize().query("model in ['M2', 'M3']"))
display(Image('results/figures/M3_S2_vianli_test_seed42.png'))

Sau khi chạy xong: tải các file `results/M3_*` (trên Drive) về và commit lên GitHub,
để bảng kết quả trong repo khớp với báo cáo. Nhận xét kết quả ở `docs/ket_qua_m3.md`.